# 模块二：中国 31 省医疗卫生资源底座构建 (核心主战场)

**执行人**：成员 4 (数据工程大师)

**核心目标**：将官方提供的“反人类宽表”重构为标准的“二维面板数据 (Panel Data)”，为后续的 DEA 效能评估模型建立极度干净的数据底座。

**包含数据集**：
1. `各省近20年卫生人员数量.csv` (省级投入端)
2. `近20年各省医疗卫生机构数量.csv` (省级投入端)
3. `近20年我国人口出生率死亡率自然增长率及平均预期寿命.csv` (国家宏观基准)

**数据工程清洗逻辑**：
1. **宽表逆透视 (Melt)**：将横向铺开的 2005-2024 年份列，全部降维拍扁成 `年份` 与 `数值` 两列。
2. **防伪脱水处理**：
   - **年份脱水**：剥离“年”字后缀，强转为 `int` 整型，打通时间序列。
   - **省份脱水 (极其关键)**：利用正则表达式暴力切除“省、市、自治区、维吾尔自治区”等冗余行政级别后缀，统一规范为核心地名（如“北京”、“内蒙古”），确保后续与信息组搜集的数据能够完美 `Merge`。
3. **分流处理**：拆分“省级面板”与“国家基准”两套表，防止宏观数据与省级数据合并时发生维度坍塌。

In [1]:
import pandas as pd
import os

print("========== 开启中国核心数据降维重组 ==========\n")

# 1. 挂载真实文件路径
path_personnel = r'D:\2026_BigData_Project\data\raw\各省近20年卫生人员数量.csv'
path_institutions = r'D:\2026_BigData_Project\data\raw\近20年各省医疗卫生机构数量.csv'
path_national_demo = r'D:\2026_BigData_Project\data\raw\近20年我国人口出生率死亡率自然增长率及平均预期寿命.csv'

# ==========================================
# 战役一：省级核心数据融合 (表1 & 表2)
# ==========================================
print("正在处理省级数据：宽表转长表 & 名称脱水...")

df_per = pd.read_csv(path_personnel)
df_inst = pd.read_csv(path_institutions)

# 宽表转长表 (Melt)
df_per_long = pd.melt(df_per, id_vars=['地区'], var_name='年份', value_name='卫生人员数量(万人)') # 假设单位是万人，可根据实际修改
df_inst_long = pd.melt(df_inst, id_vars=['地区'], var_name='年份', value_name='医疗机构数量(个)')

# 核心武器：脱水清洗函数
def clean_prov_data(df):
    # 1. 年份脱水："2024年" -> 2024 (转换为整数，方便后续排序和模型计算)
    df['年份'] = df['年份'].str.replace('年', '').astype(int)
    # 2. 省份脱水：使用正则表达式，暴力切除所有恶心的行政后缀
    # 注意：正则表达式里的 | 代表“或者”，$ 代表“匹配字符串末尾”
    df['地区'] = df['地区'].str.replace(r'(维吾尔自治区|壮族自治区|回族自治区|自治区|省|市)$', '', regex=True)
    return df

df_per_clean = clean_prov_data(df_per_long)
df_inst_clean = clean_prov_data(df_inst_long)

# 终极合并：以[地区, 年份]为防伪双主键，进行外连接关联
df_province_panel = pd.merge(df_per_clean, df_inst_clean, on=['地区', '年份'], how='outer')

# 按照地区和年份排个序，强迫症福音
df_province_panel = df_province_panel.sort_values(by=['地区', '年份']).reset_index(drop=True)

print("✅ 省级面板数据重组成功！前5行如下：")
print(df_province_panel.head())


# ==========================================
# 战役二：国家宏观基准表重构 (表3)
# ==========================================
print("\n正在处理国家级基准数据：宽转长 & 指标透视...")

df_nat = pd.read_csv(path_national_demo)

# 1. 宽表转长表
df_nat_long = pd.melt(df_nat, id_vars=['指标'], var_name='年份', value_name='数值')
# 年份脱水
df_nat_long['年份'] = df_nat_long['年份'].str.replace('年', '').astype(int)

# 2. 行列透视 (Pivot)：把“指标”这一列里的各个名字，翻转变成新的列名！
df_national_panel = df_nat_long.pivot(index='年份', columns='指标', values='数值').reset_index()

# 清理一下多余的列索引名字
df_national_panel.columns.name = None

print("✅ 国家基准数据重组成功！前5行如下：")
print(df_national_panel.head())


# ==========================================
# 战役三：成果落袋为安 (导出至 processed 文件夹)
# ==========================================
out_prov_path = r'D:\2026_BigData_Project\data\processed\China_Province_Health_Panel.csv'
out_nat_path = r'D:\2026_BigData_Project\data\processed\China_National_Demo_Panel.csv'

df_province_panel.to_csv(out_prov_path, index=False)
df_national_panel.to_csv(out_nat_path, index=False)

print(f"\n🎉 大功告成！两份核心数据已分别导出至：\n1. {out_prov_path}\n2. {out_nat_path}")

========== 开启中国核心数据降维重组 ==========

正在处理省级数据：宽表转长表 & 名称脱水...
✅ 省级面板数据重组成功！前5行如下：
   地区    年份  卫生人员数量(万人)  医疗机构数量(个)
0  上海  2005       13.20       2526
1  上海  2006       13.80       2519
2  上海  2007       15.58       2678
3  上海  2008       16.22       2822
4  上海  2009       16.95       4460

正在处理国家级基准数据：宽转长 & 指标透视...
✅ 国家基准数据重组成功！前5行如下：
     年份  人口出生率(‰)  人口死亡率(‰)  人口自然增长率(‰)  女性平均预期寿命(岁)  平均预期寿命(岁)  男性平均预期寿命(岁)
0  2005     12.40      6.51        5.89        75.25      72.95        70.83
1  2006     12.09      6.81        5.28          NaN        NaN          NaN
2  2007     12.10      6.93        5.17          NaN        NaN          NaN
3  2008     12.14      7.06        5.08          NaN        NaN          NaN
4  2009     11.95      7.08        4.87          NaN        NaN          NaN

🎉 大功告成！两份核心数据已分别导出至：
1. D:\2026_BigData_Project\data\processed\China_Province_Health_Panel.csv
2. D:\2026_BigData_Project\data\processed\China_National_Demo_Panel.csv


In [2]:
print("================ 中国核心底座深度体检报告 ================\n")

# -----------------------------------------
# 1. 体检：省级医疗面板数据 (df_province_panel)
# -----------------------------------------
print("【一】省级面板数据 (China_Province_Health_Panel) 深度体检：")

# 1.1 查缺失值
print("1. 缺失值扫描：")
print(df_province_panel.isnull().sum())

# 1.2 查省份数量与名称 (极其关键！)
unique_provinces = df_province_panel['地区'].unique()
print(f"\n2. 省份脱水核查：共识别出 {len(unique_provinces)} 个唯一地区。")
if len(unique_provinces) == 31:
    print("   ✅ 完美！刚好 31 个省/直辖市/自治区。")
else:
    print("   ❌ 警告：省份数量不是 31 个，请仔细检查下方列表是否有脏数据或重名！")
print(f"   具体名单：{unique_provinces}")

# 1.3 查异常值 (极值检测)
print("\n3. 异常数值扫描 (查看最小值 Min 是否有负数)：")
# 只看描述性统计里的最小值和最大值部分
print(df_province_panel.describe().loc[['min', 'max']])
print("-" * 50)


# -----------------------------------------
# 2. 体检：国家宏观基准表 (df_national_panel)
# -----------------------------------------
print("\n【二】国家宏观基准表 (China_National_Demo_Panel) 深度体检：")

# 2.1 查缺失值
print("1. 缺失值扫描：")
print(df_national_panel.isnull().sum())

# 2.2 查年份跨度
min_year = df_national_panel['年份'].min()
max_year = df_national_panel['年份'].max()
print(f"\n2. 时间跨度核查：从 {min_year} 年 到 {max_year} 年，共 {max_year - min_year + 1} 年。")

# 2.3 预期寿命空值说明
print("\n3. 预期寿命列空值提示：")
print("   (注：预期寿命在非普查年份出现 NaN 为正常现象，这是为了保持官方数据真实性，切勿擅自填补 0。)")

print("\n================ 体检结束 ================")

================ 中国核心底座深度体检报告 ================

【一】省级面板数据 (China_Province_Health_Panel) 深度体检：
1. 缺失值扫描：
地区            0
年份            0
卫生人员数量(万人)    0
医疗机构数量(个)     0
dtype: int64

2. 省份脱水核查：共识别出 31 个唯一地区。
   ✅ 完美！刚好 31 个省/直辖市/自治区。
   具体名单：['上海' '云南' '内蒙古' '北京' '吉林' '四川' '天津' '宁夏' '安徽' '山东' '山西' '广东' '广西' '新疆'
 '江苏' '江西' '河北' '河南' '浙江' '海南' '湖北' '湖南' '甘肃' '福建' '西藏' '贵州' '辽宁' '重庆'
 '陕西' '青海' '黑龙江']

3. 异常数值扫描 (查看最小值 Min 是否有负数)：
         年份  卫生人员数量(万人)  医疗机构数量(个)
min  2005.0        1.02     1322.0
max  2024.0      120.25    93648.0
--------------------------------------------------

【二】国家宏观基准表 (China_National_Demo_Panel) 深度体检：
1. 缺失值扫描：
年份              0
人口出生率(‰)        0
人口死亡率(‰)        0
人口自然增长率(‰)      0
女性平均预期寿命(岁)    16
平均预期寿命(岁)      16
男性平均预期寿命(岁)    16
dtype: int64

2. 时间跨度核查：从 2005 年 到 2024 年，共 20 年。

3. 预期寿命列空值提示：
   (注：预期寿命在非普查年份出现 NaN 为正常现象，这是为了保持官方数据真实性，切勿擅自填补 0。)

================ 体检结束 ================
